<a href="https://colab.research.google.com/github/Hydaspex/strait_hormuz_detector/blob/main/strait_hormuz_detection_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Intro 🛰️ Strait of Hormuz — Multi-Layer Vessel Detection Monitor
**Operational Intelligence Notebook · dynamically updated**

---

## Context

This notebook monitors vessel activity at the Strait of Hormuz using a layered detection approach designed to stay useful even when one data source is delayed, degraded, or partially blind. It combines live AIS, Sentinel-1 SAR, Sentinel-2 optical imagery, and IMF PortWatch historical transit data into a single operational workflow.

Unlike the earlier static version, this notebook now pulls historical transit series dynamically at runtime and keeps current-day indicators separate from lagged official history. That makes the dashboard easier to rerun without editing dates, fallback price points, or manually maintained daily tables.

---

## Purpose

The goal is to produce a defensible daily view of chokepoint traffic that can support market monitoring, disruption tracking, and cross-checking of dark-vessel behaviour. The workflow is structured so that historical baselines, live snapshots, and sourced fallback values are isolated and easy to update.

---

## Sensor Architecture

| Layer | Technology | What it detects | Typical latency | Notes |
|---|---|---|---|---|
| **1 · AIS** | aisstream.io WebSocket | Transponder-on vessels only | Real time | Best for live directional context |
| **2 · Sentinel-1 SAR** | Copernicus Data Space | Metal hulls including dark ships | ~6–12 hrs | Cloud independent |
| **3 · Sentinel-2 Optical** | Copernicus Data Space | Bright targets on water | ~1–5 days | Useful when cloud conditions allow |
| **4 · PortWatch** | IMF / ArcGIS REST | Official daily chokepoint transit series | Multi-day lag | Best source for historical trendline |

---

## Refactor highlights

- Historical transit history is fetched dynamically from PortWatch instead of being stored as a hardcoded table.
- Current-day live AIS can be appended as a provisional row when official history has not yet updated.
- Constants, fallback values, and source metadata are grouped into compact registries so maintenance happens in one place.
- Helper functions now separate configuration, fetch logic, and display formatting more cleanly.

---

## Maintenance model

If you need to update baseline assumptions later, you should only need to edit the configuration dictionaries in the constants section rather than hunting through multiple cells. The intended pattern is: update a registry once, and let helper functions consume it everywhere else.

#Installs & Config


In [ ]:
!pip install sentinelhub requests pandas matplotlib folium geopandas shapely -q kaleido==0.2.1 -q

import os
import json
import time
from datetime import datetime, timedelta, timezone
import requests
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from google.colab import userdata          # ← access Colab secrets
from sentinelhub import (
    SHConfig, BBox, CRS, DataCollection,
    SentinelHubRequest, MimeType, bbox_to_dimensions,
    SentinelHubCatalog,
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 10.4 MB/s eta 0:00:00


API Credentials Setup for Google Colab
Method 1 — Colab Secrets (Recommended)

The built-in Secrets panel (🔑 icon in the left sidebar) is the safest option — credentials are stored per-user in your Google account, never in the notebook file, and persist across sessions.

Step 1: Open the Secrets panel → click + Add new secret → add each key below:

* Create an account at
https://identity.dataspace.copernicus.eu/auth/realms/CDSE/login-actions/registration?client_id=sh-5f8b630b-b083-49ed-b340-b8f01ecb81c4&tab_id=5ubEOGJ16QA&client_data=eyJydSI6Imh0dHBzOi8vYnJvd3Nlci5kYXRhc3BhY2UuY29wZXJuaWN1cy5ldS8_em9vbT01JmxhdD01MC4xNjI4MiZsbmc9MjAuNzg2MTMmZGVtU291cmNlM0Q9JTIyTUFQWkVOJTIyJmNsb3VkQ292ZXJhZ2U9MzAmZGF0ZU1vZGU9U0lOR0xFIiwicnQiOiJjb2RlIiwicm0iOiJmcmFnbWVudCIsInN0IjoiZjFjZGMwY2QtYWM0My00NzM4LTg3YTgtNjVmYTZmYzBhOWFhIn0
* SENTINEL_CLIENT_ID
  * hover over Login then click  Sentinel Hub Dashboard on the pop-up menu
  * hover over Top-right hand corner username( email address )
  * click Settings  on the drop down menu
  * click +Create OAuth clients
* SENTINEL_CLIENT_SECRET	Same dashboard
* AISSTREAM_API_KEY	at https://aisstream.io/authenticate




#Values, Constants and Helpers

In [ ]:
# ── Strait of Hormuz AOI ─────────────────────────────────────────────────────
from typing import Any, Dict, List, Optional, Tuple

HORMUZ_BBOX: List[float] = [55.8, 25.8, 57.2, 26.8]
GULF_OMAN_BBOX: List[float] = [55.0, 22.5, 59.5, 26.0]
RESOLUTION: int = 10

# ═════════════════════════════════════════════════════════════════════════════
# CENTRAL CONFIG REGISTRIES
# Keep all manually maintained values here so updates happen in one place.
# ═════════════════════════════════════════════════════════════════════════════

THEME: Dict[str, str] = {
    "DARK_BG": "#0d1117",
    "PANEL_BG": "#161b22",
    "ACCENT": "#2ea9df",
    "WARN": "#e07b39",
    "DANGER": "#f05353",
    "GREEN": "#3fb950",
    "MUTED": "#7d8590",
    "TEXT": "#e6edf3",
}

MARKET_BASELINES: Dict[str, float | str] = {
    "crisis_start": "2026-02-28",
    "baseline_daily_transits": 151,
    "dark_vessel_factor": 2.5,
    "sar_fp_rate": 0.15,
    "optical_fp_rate": 0.10,
    "pre_war_risk_pct": 0.25,
    "default_current_war_risk_pct": 3.0,
}

FALLBACK_NOTES: Dict[str, str] = {
    "prices_as_of": "2026-04-09",
    "war_risk_note": "Sourced midpoint estimate; update manually when market reporting changes.",
    "price_note": "Fallback prices are used only when Yahoo Finance is unavailable.",
}

COMMODITY_REGISTRY: Dict[str, Dict[str, Any]] = {
    "BZ=F": {
        "label": "Brent Crude",
        "unit": "$/bbl",
        "pre": 72.00,
        "fallback": 96.76,
        "fallback_as_of": "2026-04-09",
    },
    "CL=F": {
        "label": "WTI Crude",
        "unit": "$/bbl",
        "pre": 65.00,
        "fallback": 97.05,
        "fallback_as_of": "2026-04-09",
    },
    "TTF=F": {
        "label": "EU Gas (TTF)",
        "unit": "€/MWh",
        "pre": 30.00,
        "fallback": 50.00,
        "fallback_as_of": "2026-04-09",
    },
    "RB=F": {
        "label": "US Gasoline",
        "unit": "$/gal",
        "pre": 2.97,
        "fallback": 2.93,
        "fallback_as_of": "2026-04-08",
    },
}

PORTWATCH_CONFIG: Dict[str, Any] = {
    "url": "https://services9.arcgis.com/weJ1QsnbMYJlCHdG/arcgis/rest/services/Daily_Chokepoints_Data/FeatureServer/0/query",
    "portid": "chokepoint6",
    "history_days": 180,
    "timeout_sec": 30,
}

# ── Flattened aliases preserved for downstream notebook compatibility ─────────
DARK_BG: str = THEME["DARK_BG"]
PANEL_BG: str = THEME["PANEL_BG"]
ACCENT: str = THEME["ACCENT"]
WARN: str = THEME["WARN"]
DANGER: str = THEME["DANGER"]
GREEN: str = THEME["GREEN"]
MUTED: str = THEME["MUTED"]
TEXT: str = THEME["TEXT"]

CRISIS_START: datetime = datetime.fromisoformat(str(MARKET_BASELINES["crisis_start"]))
TODAY: datetime = datetime.today()
DAYS_DISRUPTED: int = (TODAY - CRISIS_START).days
BASELINE_DAILY: float = float(MARKET_BASELINES["baseline_daily_transits"])
DARK_VESSEL_FACTOR: float = float(MARKET_BASELINES["dark_vessel_factor"])
SAR_FP_RATE: float = float(MARKET_BASELINES["sar_fp_rate"])
OPTICAL_FP_RATE: float = float(MARKET_BASELINES["optical_fp_rate"])
PRE_WAR_RISK: float = float(MARKET_BASELINES["pre_war_risk_pct"])
DEFAULT_CURRENT_WAR_RISK: float = float(MARKET_BASELINES["default_current_war_risk_pct"])


def commodity_rows() -> List[Dict[str, Any]]:
    """Return commodity registry entries as row dictionaries.

    Returns:
        List[Dict[str, Any]]: Commodity configuration rows with ticker included.
    """
    rows: List[Dict[str, Any]] = []
    for ticker, cfg in COMMODITY_REGISTRY.items():
        row: Dict[str, Any] = {"ticker": ticker}
        row.update(cfg)
        rows.append(row)
    return rows


def format_fetch_status(is_live: bool) -> str:
    """Format a human-readable data freshness flag.

    Args:
        is_live: Whether the upstream fetch returned a live market value.

    Returns:
        str: A status label indicating live or fallback mode.
    """
    return "✅ live" if is_live else "⚠ fallback"


def pct_change(current: float, baseline: float) -> float:
    """Compute percent change from a baseline value.

    Args:
        current: Current observed value.
        baseline: Baseline reference value.

    Returns:
        float: Percent change relative to baseline, or NaN when baseline is invalid.
    """
    if baseline in (None, 0):
        return np.nan
    return (current - baseline) / baseline * 100


def _yahoo_fetch(ticker: str) -> Tuple[float, str]:
    """Fetch a commodity quote from Yahoo Finance's chart endpoint.

    Args:
        ticker: Yahoo Finance futures ticker.

    Returns:
        Tuple[float, str]: The fetched price and the market timestamp string.

    Raises:
        requests.RequestException: If the HTTP request fails.
        KeyError: If the response schema changes.
        IndexError: If the expected result array is missing.
        ValueError: If the response cannot be converted to a float.
    """
    url: str = f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}"
    headers: Dict[str, str] = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64)"}
    r = requests.get(url, headers=headers, timeout=8)
    r.raise_for_status()
    meta: Dict[str, Any] = r.json()["chart"]["result"][0]["meta"]
    price: Any = meta.get("regularMarketPrice") or meta.get("previousClose")
    ts: str = datetime.utcfromtimestamp(meta.get("regularMarketTime", 0)).strftime("%Y-%m-%d %H:%MZ")
    return float(price), ts


def fetch_commodity_price(ticker: str) -> Tuple[float, str, bool]:
    """Fetch a commodity price with registry-based fallback handling.

    Args:
        ticker: Yahoo Finance futures ticker defined in ``COMMODITY_REGISTRY``.

    Returns:
        Tuple[float, str, bool]: The resolved price, timestamp label, and a
        boolean indicating whether the value is live.
    """
    cfg: Dict[str, Any] = COMMODITY_REGISTRY[ticker]
    try:
        price, ts = _yahoo_fetch(ticker)
        return price, ts, True
    except Exception as e:
        print(f"  ⚠ Yahoo fetch failed [{ticker}]: {e} — using fallback {cfg['fallback']} ({cfg['fallback_as_of']})")
        return float(cfg["fallback"]), f"fallback ({cfg['fallback_as_of']})", False


def fetch_all_commodity_prices() -> pd.DataFrame:
    """Resolve all configured commodity prices into a tidy dataframe.

    Returns:
        pd.DataFrame: Commodity quotes with baseline, current value, percent
        change, timestamp, and fallback metadata.
    """
    records: List[Dict[str, Any]] = []
    for row in commodity_rows():
        price, ts, is_live = fetch_commodity_price(row["ticker"])
        records.append({
            "ticker": row["ticker"],
            "label": row["label"],
            "unit": row["unit"],
            "pre": row["pre"],
            "current": round(price, 4),
            "pct_change": round(pct_change(price, row["pre"]), 1),
            "timestamp": ts,
            "is_live": is_live,
            "fallback": row["fallback"],
            "fallback_as_of": row["fallback_as_of"],
        })
    return pd.DataFrame(records)


def fetch_brent_price() -> Tuple[float, str]:
    """Fetch the Brent price while preserving the notebook's legacy interface.

    Returns:
        Tuple[float, str]: The Brent price and timestamp string.
    """
    price, ts, _ = fetch_commodity_price("BZ=F")
    return price, ts


def get_war_risk_multiplier(current_rate_pct: float = DEFAULT_CURRENT_WAR_RISK) -> Tuple[str, str]:
    """Calculate the war-risk premium multiplier versus baseline.

    Args:
        current_rate_pct: Current additional war-risk premium percentage.

    Returns:
        Tuple[str, str]: A display multiplier string and supporting subtitle text.
    """
    multiplier: float = current_rate_pct / PRE_WAR_RISK
    return f"{multiplier:.0f}×", f"~{current_rate_pct}% vs {PRE_WAR_RISK}% baseline"


print("Fetching live commodity prices from Yahoo Finance...")
commodity_df = fetch_all_commodity_prices()

commodity_labels = commodity_df["label"].tolist()
commodity_pre = commodity_df["pre"].tolist()
commodity_current = commodity_df["current"].tolist()
commodity_pct = commodity_df["pct_change"].tolist()
commodity_units = commodity_df["unit"].tolist()
commodity_ts = commodity_df["timestamp"].tolist()
commodity_live = commodity_df["is_live"].tolist()

for _, row in commodity_df.iterrows():
    print(
        f"  {row['label']:16s}: {row['current']:>8.2f} {row['unit']}  "
        f"({row['pct_change']:+.1f}% vs pre-crisis)  [{row['timestamp']}]  {format_fetch_status(bool(row['is_live']))}"
    )

FETCH_TIME: str = datetime.utcnow().strftime("%Y-%m-%d %H:%MZ")

brent_row = commodity_df.loc[commodity_df["ticker"] == "BZ=F"].iloc[0]
wti_row = commodity_df.loc[commodity_df["ticker"] == "CL=F"].iloc[0]
ttf_row = commodity_df.loc[commodity_df["ticker"] == "TTF=F"].iloc[0]
gasoline_row = commodity_df.loc[commodity_df["ticker"] == "RB=F"].iloc[0]

brent_price, brent_ts = float(brent_row["current"]), str(brent_row["timestamp"])
wti_price, wti_ts = float(wti_row["current"]), str(wti_row["timestamp"])
ttf_price, ttf_ts = float(ttf_row["current"]), str(ttf_row["timestamp"])
gasoline_price, gasoline_ts = float(gasoline_row["current"]), str(gasoline_row["timestamp"])

brent_pct_change = float(brent_row["pct_change"])
wti_pct_change = float(wti_row["pct_change"])
ttf_pct_change = float(ttf_row["pct_change"])
gasoline_pct_change = float(gasoline_row["pct_change"])

brent_label = f"${brent_price:.2f}" + ("" if bool(brent_row["is_live"]) else "*")
brent_sub = f"{brent_pct_change:+.0f}% vs pre-crisis (as of {brent_ts})"
brent_color = DANGER if brent_pct_change > 40 else WARN

war_risk_label, war_risk_sub = get_war_risk_multiplier(DEFAULT_CURRENT_WAR_RISK)

print(f"{'─'*58}")
print(f"  Strait of Hormuz Vessel Tracker  —  {TODAY.strftime('%Y-%m-%d')}")
print(f"{'─'*58}")
print(f"  Crisis start     : {CRISIS_START.date()}")
print(f"  Days disrupted   : {DAYS_DISRUPTED}")
print(f"  Transit baseline : ~{BASELINE_DAILY} vessels/day")
print(f"{'─'*58}")
print(f"  {'Commodity':<18} {'Pre':>8} {'Current':>10} {'Δ%':>8}")
print(f"  {'─'*46}")
for _, row in commodity_df.iterrows():
    live_flag = "" if bool(row["is_live"]) else "*"
    print(f"  {row['label']:<18} {row['pre']:>7.2f} {row['unit'][:5]:<5} {row['current']:>8.2f}{live_flag} {row['pct_change']:>+7.1f}%")
print(f"{'─'*58}")
print(f"  Brent KPI        : {brent_label}  {brent_sub}")
print(f"  War risk KPI     : {war_risk_label}  ({war_risk_sub})")
print(f"  Prices fetched   : {FETCH_TIME}")
print(f"  * = cached fallback price")
print(f"  ⚠ {FALLBACK_NOTES['war_risk_note']}")

Fetching live commodity prices from Yahoo Finance...
  Brent Crude     :    96.61 $/bbl  (+34.2% vs pre-crisis)  [2026-04-23 12:02Z]  ✅ live
  WTI Crude       :    93.09 $/bbl  (+43.2% vs pre-crisis)  [2026-04-23 12:02Z]  ✅ live
  EU Gas (TTF)    :    45.56 €/MWh  (+51.9% vs pre-crisis)  [2026-04-23 07:42Z]  ✅ live
  US Gasoline     :     3.23 $/gal  (+8.7% vs pre-crisis)  [2026-04-23 12:02Z]  ✅ live
──────────────────────────────────────────────────────────
  Strait of Hormuz Vessel Tracker  —  2026-04-23
──────────────────────────────────────────────────────────
  Crisis start     : 2026-02-28
  Days disrupted   : 54
  Transit baseline : ~151.0 vessels/day
──────────────────────────────────────────────────────────
  Commodity               Pre    Current       Δ%
  ──────────────────────────────────────────────
  Brent Crude          72.00 $/bbl    96.61   +34.2%
  WTI Crude            65.00 $/bbl    93.09   +43.2%
  EU Gas (TTF)         30.00 €/MWh    45.56   +51.9%
  US Gasoline   

In [ ]:
# Historical Transit Time-Series
# Registry-driven PortWatch history + live AIS overlay
from typing import Any, Dict, Optional

PORTWATCH_SCHEMA: Dict[str, Any] = {
    "date_field": "date",
    "numeric_fields": [
        "n_total", "n_tanker", "n_container",
        "n_dry_bulk", "n_general_cargo", "n_roro",
        "capacity_tanker", "capacity",
    ],
    "history_value_field": "n_total",
}

HISTORY_NOTES: Dict[str, str] = {
    "pre_crisis": "Pre-crisis",
    "crisis": "Crisis period",
    "observed": "PortWatch observed",
    "live_overlay": "Live AIS snapshot; PortWatch pending",
    "empty": "No PortWatch history returned.",
}

DASHBOARD_CONFIG: Dict[str, Any] = {
    "title": "Strait of Hormuz Transit History",
    "height": 520,
    "show_live_overlay": True,
    "history_series_name": "Daily transits",
    "live_series_name": "Live AIS snapshot",
    "history_fill": "rgba(46,169,223,0.10)",
    "history_days_default": PORTWATCH_CONFIG["history_days"],
}


def utc_today() -> pd.Timestamp:
    """Return today's UTC date normalized to midnight.

    Returns:
        pd.Timestamp: A timezone-naive UTC-normalized timestamp.
    """
    return pd.Timestamp.utcnow().normalize().tz_localize(None)


def build_portwatch_params(config: Dict[str, Any]) -> Dict[str, Any]:
    """Build ArcGIS query parameters for the PortWatch endpoint.

    Args:
        config: PortWatch runtime configuration dictionary.

    Returns:
        Dict[str, Any]: Querystring parameters for the ArcGIS REST request.
    """
    return {
        "where": f"portid='{config['portid']}'",
        "outFields": ",".join([PORTWATCH_SCHEMA["date_field"], *PORTWATCH_SCHEMA["numeric_fields"]]),
        "maxRecordCountFactor": 5,
        "outSR": 4326,
        "orderByFields": f"{PORTWATCH_SCHEMA['date_field']} ASC",
        "f": "json",
    }


def normalize_portwatch_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize raw PortWatch attributes into an analysis-ready dataframe.

    Args:
        df: Raw dataframe created from PortWatch feature attributes.

    Returns:
        pd.DataFrame: Sorted dataframe with typed numeric fields and baseline percentages.
    """
    out = df.copy()
    date_field = PORTWATCH_SCHEMA["date_field"]
    out[date_field] = pd.to_datetime(out[date_field], unit="ms").dt.normalize().dt.tz_localize(None)
    for c in PORTWATCH_SCHEMA["numeric_fields"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    out = out.sort_values(date_field).reset_index(drop=True)
    out["pct_of_baseline"] = (out[PORTWATCH_SCHEMA["history_value_field"]] / BASELINE_DAILY * 100).round(1)
    out["source"] = "IMF PortWatch"
    return out


def fetch_portwatch_history(config: Dict[str, Any] = PORTWATCH_CONFIG) -> pd.DataFrame:
    """Fetch the full PortWatch history for the configured chokepoint.

    Args:
        config: PortWatch runtime configuration.

    Returns:
        pd.DataFrame: Normalized PortWatch history, or an empty dataframe.

    Raises:
        RuntimeError: If the ArcGIS API returns an application-level error.
        requests.RequestException: If the HTTP request fails.
    """
    r = requests.get(config["url"], params=build_portwatch_params(config), timeout=config["timeout_sec"])
    r.raise_for_status()
    payload = r.json()
    if "error" in payload:
        raise RuntimeError(f"PortWatch API error: {payload['error']}")
    features = payload.get("features", [])
    if not features:
        return pd.DataFrame()
    df = pd.DataFrame([f["attributes"] for f in features])
    return normalize_portwatch_frame(df)


def apply_history_labels(df: pd.DataFrame, crisis_start: Optional[Any] = None) -> pd.DataFrame:
    """Add notebook-specific analysis labels to PortWatch history.

    Args:
        df: Normalized PortWatch dataframe.
        crisis_start: Optional crisis start value convertible to ``pd.Timestamp``.

    Returns:
        pd.DataFrame: History dataframe with note labels and overlay columns.
    """
    out = df.copy()
    out["vessels_portwatch"] = out[PORTWATCH_SCHEMA["history_value_field"]]
    out["vessels_ais"] = np.nan
    out["vessels_sar_est"] = np.nan
    out["notes"] = HISTORY_NOTES["observed"]
    if crisis_start is not None:
        crisis_start_ts = pd.Timestamp(crisis_start).normalize()
        out.loc[out["date"] < crisis_start_ts, "notes"] = HISTORY_NOTES["pre_crisis"]
        out.loc[out["date"] >= crisis_start_ts, "notes"] = HISTORY_NOTES["crisis"]
    return out


def append_live_ais_overlay(
    df: pd.DataFrame,
    ais_live_count: Optional[int] = None,
    today: Optional[Any] = None,
    enabled: bool = True,
) -> pd.DataFrame:
    """Append a provisional live AIS row when official history is lagging.

    Args:
        df: History dataframe.
        ais_live_count: Current AIS count in the chokepoint.
        today: Optional override for today's date.
        enabled: Whether live overlay behaviour is enabled.

    Returns:
        pd.DataFrame: History dataframe with optional live AIS overlay row.
    """
    if not enabled or ais_live_count is None or df.empty:
        return df
    today_ts = utc_today() if today is None else pd.Timestamp(today).normalize()
    latest_pw_date = df["date"].max()
    if latest_pw_date >= today_ts:
        return df
    live_row = {
        "date": today_ts,
        "n_total": np.nan,
        "n_tanker": np.nan,
        "n_container": np.nan,
        "n_dry_bulk": np.nan,
        "n_general_cargo": np.nan,
        "n_roro": np.nan,
        "capacity_tanker": np.nan,
        "capacity": np.nan,
        "pct_of_baseline": round((ais_live_count / BASELINE_DAILY) * 100, 1),
        "source": "AIS live snapshot",
        "vessels_portwatch": np.nan,
        "vessels_ais": ais_live_count,
        "vessels_sar_est": np.nan,
        "notes": HISTORY_NOTES["live_overlay"],
    }
    return pd.concat([df, pd.DataFrame([live_row])], ignore_index=True)


def build_dynamic_history(
    include_live_ais: bool = True,
    history_days: Optional[int] = None,
    ais_live_count: Optional[int] = None,
    crisis_start: Optional[Any] = None,
) -> pd.DataFrame:
    """Build the notebook's canonical historical transit dataframe.

    Args:
        include_live_ais: Whether to append a live AIS overlay row.
        history_days: Number of trailing days to retain.
        ais_live_count: Current AIS vessel count for the strait.
        crisis_start: Optional crisis start date for note labeling.

    Returns:
        pd.DataFrame: Final history dataframe sorted by date.
    """
    resolved_history_days = history_days or DASHBOARD_CONFIG["history_days_default"]
    today = utc_today()
    df = fetch_portwatch_history(PORTWATCH_CONFIG)
    if df.empty:
        return df
    cutoff = today - pd.Timedelta(days=resolved_history_days)
    df = df[df["date"] >= cutoff].copy()
    df = apply_history_labels(df, crisis_start=crisis_start)
    df = append_live_ais_overlay(df, ais_live_count=ais_live_count, today=today, enabled=include_live_ais)
    return df.sort_values("date").reset_index(drop=True)


ais_live_count = len(hormuz_vessels) if 'hormuz_vessels' in globals() else None
transit_history = build_dynamic_history(
    include_live_ais=DASHBOARD_CONFIG["show_live_overlay"],
    history_days=DASHBOARD_CONFIG["history_days_default"],
    ais_live_count=ais_live_count,
    crisis_start=CRISIS_START,
)

if not transit_history.empty:
    display_cols = [c for c in ["date", "vessels_portwatch", "vessels_ais", "pct_of_baseline", "notes", "source"] if c in transit_history.columns]
    print(transit_history.tail(10)[display_cols].to_string(index=False))
else:
    print(HISTORY_NOTES["empty"])

      date  vessels_portwatch  vessels_ais  pct_of_baseline         notes        source
2026-04-10                  5          NaN              3.3 Crisis period IMF PortWatch
2026-04-11                 11          NaN              7.3 Crisis period IMF PortWatch
2026-04-12                  8          NaN              5.3 Crisis period IMF PortWatch
2026-04-13                 11          NaN              7.3 Crisis period IMF PortWatch
2026-04-14                 10          NaN              6.6 Crisis period IMF PortWatch
2026-04-15                 10          NaN              6.6 Crisis period IMF PortWatch
2026-04-16                  6          NaN              4.0 Crisis period IMF PortWatch
2026-04-17                 14          NaN              9.3 Crisis period IMF PortWatch
2026-04-18                 29          NaN             19.2 Crisis period IMF PortWatch
2026-04-19                  4          NaN              2.6 Crisis period IMF PortWatch


#AIS Vessel Detection


In [ ]:
# ── Install nest_asyncio once ─────────────────────────────────────────────────
import nest_asyncio
nest_asyncio.apply()
import asyncio
import websockets
from typing import Any, Dict, List, Sequence

AISSTREAM_API_KEY: str = userdata.get("AISSTREAM_API_KEY")
AIS_VESSEL_TYPES: Dict[int, str] = {
    30: "Fishing", 31: "Towing",
    36: "Sailing", 37: "Pleasure craft",
    60: "Passenger", 70: "Cargo",
    71: "Cargo – Hazmat A", 72: "Cargo – Hazmat B",
    73: "Cargo – Hazmat C", 74: "Cargo – Hazmat D",
    80: "Tanker", 81: "Tanker – Hazmat A",
    82: "Tanker – Hazmat B", 83: "Tanker – Hazmat C",
    84: "Tanker – Hazmat D", 35: "Military",
    55: "Law Enforcement",
}


def bbox_to_aisstream(bbox: Sequence[float]) -> List[List[float]]:
    """Convert a lon/lat bounding box into AISStream format.

    Args:
        bbox: Bounding box in ``[lon_min, lat_min, lon_max, lat_max]`` order.

    Returns:
        List[List[float]]: AISStream bounding box in ``[[lat_min, lon_min], [lat_max, lon_max]]`` format.
    """
    lon_min, lat_min, lon_max, lat_max = bbox
    return [[lat_min, lon_min], [lat_max, lon_max]]


async def fetch_ais_snapshot(
    bbox: Sequence[float],
    timeout_sec: int = 300,
    max_vessels: int = 500,
) -> List[Dict[str, Any]]:
    """Fetch a live AIS vessel snapshot from AISStream.

    Args:
        bbox: Bounding box in lon/lat order.
        timeout_sec: Maximum time window to listen for AIS messages.
        max_vessels: Maximum number of unique vessels to collect.

    Returns:
        List[Dict[str, Any]]: Deduplicated vessel records keyed by MMSI.
    """
    vessels: Dict[str, Dict[str, Any]] = {}
    subscribe_msg: str = json.dumps({
        "APIKey": AISSTREAM_API_KEY,
        "BoundingBoxes": [bbox_to_aisstream(bbox)],
        "FilterMessageTypes": ["PositionReport", "ShipStaticData"],
    })

    try:
        async with websockets.connect(
            "wss://stream.aisstream.io/v0/stream",
            open_timeout=15,
            close_timeout=5,
            ping_interval=20,
            ping_timeout=10,
        ) as ws:
            await ws.send(subscribe_msg)
            deadline: float = asyncio.get_event_loop().time() + timeout_sec
            last_count: int = 0

            while asyncio.get_event_loop().time() < deadline and len(vessels) < max_vessels:
                try:
                    raw: str = await asyncio.wait_for(ws.recv(), timeout=10.0)
                    msg: Dict[str, Any] = json.loads(raw)
                    if msg.get("MessageType") == "PositionReport":
                        pos: Dict[str, Any] = msg["Message"]["PositionReport"]
                        meta: Dict[str, Any] = msg.get("MetaData", {})
                        mmsi: str = str(pos.get("UserID", ""))
                        vessels[mmsi] = {
                            "mmsi": mmsi,
                            "lat": pos.get("Latitude", 0.0),
                            "lon": pos.get("Longitude", 0.0),
                            "cog": pos.get("Cog", 0.0),
                            "sog": pos.get("Sog", 0.0),
                            "heading": pos.get("TrueHeading", 511),
                            "name": meta.get("ShipName", "Unknown").strip(),
                            "type": meta.get("ShipType", 0),
                            "flag": meta.get("MMSI_CountryCode", ""),
                            "nav_status": pos.get("NavigationalStatus", 0),
                        }
                    if len(vessels) // 50 > last_count // 50:
                        elapsed: int = int(timeout_sec - (deadline - asyncio.get_event_loop().time()))
                        print(f"    ...{len(vessels)} vessels ({elapsed}s elapsed)")
                        last_count = len(vessels)
                except asyncio.TimeoutError:
                    continue
    except Exception as e:
        print(f"  AISStream WebSocket error: {e}")

    return list(vessels.values())


def fetch_ais_sync(
    bbox: Sequence[float],
    timeout_sec: int = 30,
    label: str = "zone",
) -> List[Dict[str, Any]]:
    """Synchronously fetch AIS data in notebook environments.

    Args:
        bbox: Bounding box in lon/lat order.
        timeout_sec: Number of seconds to wait for messages.
        label: Human-readable label used in error output.

    Returns:
        List[Dict[str, Any]]: Vessel snapshot records.
    """
    try:
        loop = asyncio.get_event_loop()
        return loop.run_until_complete(fetch_ais_snapshot(bbox, timeout_sec=timeout_sec))
    except Exception as e:
        print(f"  fetch_ais_sync error for {label}: {e}")
        return []


def decode_vessel_type(type_code: Any) -> str:
    """Decode an AIS vessel type code to a human-readable label.

    Args:
        type_code: Numeric AIS ship type code or fallback value.

    Returns:
        str: Human-readable vessel type label.
    """
    if isinstance(type_code, int):
        return AIS_VESSEL_TYPES.get(type_code, f"Type-{type_code}")
    return str(type_code)


def classify_transit(vessel: Dict[str, Any]) -> str:
    """Infer vessel direction from course over ground and speed.

    Args:
        vessel: AIS vessel record containing ``cog`` and ``sog`` fields.

    Returns:
        str: Transit classification label.
    """
    cog: float = float(vessel.get("cog", 0))
    sog: float = float(vessel.get("sog", 0))
    if sog < 1.0:
        return "Anchored/Drifting"
    if 50 <= cog <= 130:
        return "Outbound"
    if 230 <= cog <= 310:
        return "Inbound"
    return "Transit (unclear dir.)"


print("Fetching AIS vessels in Strait of Hormuz (30s window)...")
hormuz_vessels: List[Dict[str, Any]] = fetch_ais_sync(HORMUZ_BBOX, timeout_sec=30, label="Hormuz")

print("Fetching AIS vessels in Gulf of Oman (staging area, 30s window)...")
staging_vessels: List[Dict[str, Any]] = fetch_ais_sync(GULF_OMAN_BBOX, timeout_sec=30, label="Gulf of Oman")

for vessel in hormuz_vessels:
    vessel["zone"] = "Strait (Active)"
    vessel["type_name"] = decode_vessel_type(vessel.get("type", 0))
    vessel["direction"] = classify_transit(vessel)

for vessel in staging_vessels:
    vessel["zone"] = "Gulf of Oman (Staging)"
    vessel["type_name"] = decode_vessel_type(vessel.get("type", 0))
    vessel["direction"] = classify_transit(vessel)

all_vessels: List[Dict[str, Any]] = hormuz_vessels + staging_vessels
df_ais: pd.DataFrame = pd.DataFrame(all_vessels) if all_vessels else pd.DataFrame()

print(f"{'='*55}")
print(f"  AIS snapshot: {datetime.utcnow().strftime('%Y-%m-%d %H:%MZ')}")
print(f"  Vessels in strait   : {len(hormuz_vessels)}")
print(f"  Vessels in staging  : {len(staging_vessels)}")
print("  AIS RELIABILITY NOTE: Counts are lower bounds.")
print("  Many vessels have disabled transponders.")
print(f"{'='*55}")
if not df_ais.empty and "name" in df_ais.columns:
    cols = [c for c in ["name", "type_name", "zone", "direction", "sog", "flag"] if c in df_ais.columns]
    print(df_ais[cols].to_string(index=False))

Fetching AIS vessels in Strait of Hormuz (30s window)...
Fetching AIS vessels in Gulf of Oman (staging area, 30s window)...
  AIS snapshot: 2026-04-23 12:15Z
  Vessels in strait   : 0
  Vessels in staging  : 0
  AIS RELIABILITY NOTE: Counts are lower bounds.
  Many vessels have disabled transponders.


#Sentinel-1 SAR Vessel Detection (Dark Ships)


In [ ]:
# --------------------------------------------------------------
# CDSE Sentinel Hub config (EXPLICIT PARAMETERS ONLY)
# --------------------------------------------------------------


# ---- Pull client ID/secret from Colab Secrets (sidebar ❅) ----
SH_CLIENT_ID     = userdata.get('SH_CLIENT_ID')
SH_CLIENT_SECRET = userdata.get('SH_CLIENT_SECRET')

# ---- Build SHConfig with EXPLICIT CDSE PARAMETERS (highest precedence) ----
config = SHConfig(
    # Credentials (from your secrets)
    sh_client_id     = SH_CLIENT_ID,
    sh_client_secret = SH_CLIENT_SECRET,
    # CDSE-specific endpoints (explicitly set - highest precedence)
    sh_token_url     = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
    sh_base_url      = "https://sh.dataspace.copernicus.eu",
    # Critical: instance_id must be empty string to prevent legacy fallback
    instance_id      = "",
    # Optional: use_defaults=True to ignore any config file/profile
    use_defaults     = False
)


#   DEBUG: Verify the values we explicitly set
print("🔧 CDSE Config Verification (explicit parameters):")
print(f"   sh_base_url : {config.sh_base_url}")
print(f"   sh_token_url: {config.sh_token_url}")
print(f"   instance_id : {repr(config.instance_id)}")
print(f"   client ID   : {config.sh_client_id[:8]}..." if config.sh_client_id else "   client ID   : NOT SET")



# -----------------------------------------------------------------
# TEST AUTHENTICATION WITH SentinelHubCatalog
# -----------------------------------------------------------------
print("\n↔ Testing CDSE authentication...")
try:
    catalog = SentinelHubCatalog(config=config)
    collections = catalog.get_collections()
    print(f"✅ Auth OK: Found {len(collections)} collections")

    # Check for Sentinel-1 GRD (what we need for SAR)
    s1_grd_available = any(c['id'] == 'sentinel-1-grd' for c in collections)
    print(f"Sentinel-1 GRD available: {s1_grd_available}")

    if not s1_grd_available:
        print("⚠️ Warning: Sentinel-1 GRD collection not found in available collections")

except Exception as e:
    print(f"❌ Auth failed: {e}")
    print("\n↔ Troubleshooting tips:")
    print("   1. Verify your CDSE credentials in Colab Secrets match those from")
    print(      "      https://dataspace.copernicus.eu/dashboard → OAuth clients")
    print("   2. Ensure secrets are named exactly: SH_CLIENT_ID and SH_CLIENT_SECRET")
    print("   3. Check that your OAuth client has 'Process API' access enabled")
    print("   4. Try generating new credentials in the CDSE dashboard")
    # Don't proceed if auth fails
    raise SystemExit("Authentication failed - cannot proceed with data fetching")

🔧 CDSE Config Verification (explicit parameters):
   sh_base_url : https://sh.dataspace.copernicus.eu
   sh_token_url: https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token
   instance_id : ''
   client ID   : sh-cf7b5...

↔ Testing CDSE authentication...
✅ Auth OK: Found 10 collections
Sentinel-1 GRD available: True


In [ ]:
# --------------------------------------------------------------
# CDSE Sentinel Hub config (EXPLICIT PARAMETERS ONLY)
# --------------------------------------------------------------
from typing import Any, Dict, List, Sequence, Tuple

import scipy.ndimage as ndi
from scipy.spatial.distance import cdist
from skimage.measure import label, regionprops
from shapely.geometry import LineString, Point, Polygon
from shapely.ops import unary_union

SH_CLIENT_ID: str = userdata.get('SH_CLIENT_ID')
SH_CLIENT_SECRET: str = userdata.get('SH_CLIENT_SECRET')

config = SHConfig(
    sh_client_id=SH_CLIENT_ID,
    sh_client_secret=SH_CLIENT_SECRET,
    sh_token_url="https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
    sh_base_url="https://sh.dataspace.copernicus.eu",
    instance_id="",
    use_defaults=False,
)

print("🔧 CDSE Config Verification (explicit parameters):")
print(f"  sh_base_url : {config.sh_base_url}")
print(f"  sh_token_url: {config.sh_token_url}")
print(f"  instance_id : {repr(config.instance_id)}")
print(f"  client ID   : {config.sh_client_id[:8]}..." if config.sh_client_id else "  client ID   : NOT SET")

print("↔ Testing CDSE authentication...")
try:
    catalog = SentinelHubCatalog(config=config)
    collections = catalog.get_collections()
    print(f"✅ Auth OK: Found {len(collections)} collections")
    s1_grd_available: bool = any(c['id'] == 'sentinel-1-grd' for c in collections)
    print(f"Sentinel-1 GRD available: {s1_grd_available}")
    if not s1_grd_available:
        print("⚠️ Warning: Sentinel-1 GRD collection not found in available collections")
except Exception as e:
    print(f"❌ Auth failed: {e}")
    raise SystemExit("Authentication failed - cannot proceed with data fetching")


def fetch_s1_sar_for_hormuz(
    bbox_coords: Sequence[float],
    days_back: int = 3,
    label: str = "Hormuz",
) -> List[np.ndarray]:
    """Fetch Sentinel-1 SAR imagery for the target bounding box.

    Args:
        bbox_coords: Bounding box in lon/lat order.
        days_back: Number of trailing days to search for imagery.
        label: Human-readable label for logging.

    Returns:
        List[np.ndarray]: Retrieved SAR scenes as arrays.
    """
    end_dt: datetime = datetime.utcnow()
    start_dt: datetime = end_dt - timedelta(days=days_back)
    bbox_sh = BBox(bbox=bbox_coords, crs=CRS.WGS84)
    size: Tuple[int, int] = bbox_to_dimensions(bbox_sh, resolution=RESOLUTION)
    max_px: int = 1024
    if size[0] > max_px or size[1] > max_px:
        scale: float = max(size[0], size[1]) / max_px
        size = (int(size[0] / scale), int(size[1] / scale))

    evalscript: str = """
    //VERSION=3
    function setup() {
      return {
        input: ["VV"],
        output: { bands: 1, sampleType: "FLOAT32" }
      };
    }
    function toDb(linear) {
      return (10 * Math.log(linear)) / Math.LN10;
    }
    function evaluatePixel(samples) {
      return [toDb(samples.VV)];
    }
    """

    s1_cdse = DataCollection.SENTINEL1_IW.define_from(
        "s1iw_cdse",
        service_url=config.sh_base_url,
    )
    req = SentinelHubRequest(
        evalscript=evalscript,
        input_data=[SentinelHubRequest.input_data(
            data_collection=s1_cdse,
            time_interval=(start_dt.strftime("%Y-%m-%d"), end_dt.strftime("%Y-%m-%d")),
            other_args={"processing": {"backCoeff": "GAMMA0_TERRAIN", "orthorectify": True}},
        )],
        responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
        bbox=bbox_sh,
        size=size,
        config=config,
    )
    imgs: List[np.ndarray] = req.get_data()
    print(f"  S1 SAR scenes fetched for {label}: {len(imgs)}")
    return imgs



# -------------------------------------------------------------------
# Strait lane geometry (approximate traffic-separation scheme)
# Defined in lon/lat, then rasterized into image space.
# -------------------------------------------------------------------
INBOUND_LANE_LONLAT = [
    (56.50, 26.50),
    (56.60, 26.48),
    (56.80, 26.45),
    (57.00, 26.42),
    (57.20, 26.38),
    (57.40, 26.33),
]

OUTBOUND_LANE_LONLAT = [
    (55.90, 26.18),
    (56.10, 26.22),
    (56.30, 26.28),
    (56.50, 26.32),
    (56.70, 26.28),
    (56.90, 26.22),
]

def lonlat_to_pixel(lon, lat, bbox, shape):
    lon_min, lat_min, lon_max, lat_max = bbox
    h, w = shape
    x = (lon - lon_min) / (lon_max - lon_min) * (w - 1)
    y = (lat_max - lat) / (lat_max - lat_min) * (h - 1)
    return x, y

def polyline_to_pixels(points_lonlat, bbox, shape):
    return [lonlat_to_pixel(lon, lat, bbox, shape) for lon, lat in points_lonlat]

def make_lane_mask(shape, bbox, lane_half_width_px=18):
    h, w = shape

    inbound_px = polyline_to_pixels(INBOUND_LANE_LONLAT, bbox, shape)
    outbound_px = polyline_to_pixels(OUTBOUND_LANE_LONLAT, bbox, shape)

    inbound_line = LineString(inbound_px)
    outbound_line = LineString(outbound_px)

    lane_union = unary_union([
        inbound_line.buffer(lane_half_width_px),
        outbound_line.buffer(lane_half_width_px),
    ])

    yy, xx = np.mgrid[0:h, 0:w]
    coords = np.column_stack([xx.ravel(), yy.ravel()])

    mask = np.zeros((h, w), dtype=bool)
    for i, (x, y) in enumerate(coords):
        if lane_union.contains(Point(float(x), float(y))):
            mask.ravel()[i] = True

    return mask, inbound_line, outbound_line

def make_border_exclusion_mask(shape, border_px=20):
    h, w = shape
    mask = np.ones((h, w), dtype=bool)
    mask[:border_px, :] = False
    mask[-border_px:, :] = False
    mask[:, :border_px] = False
    mask[:, -border_px:] = False
    return mask

def region_distance_to_lines(region, lines):
    cy, cx = region.centroid
    p = np.array([[cx, cy]])
    min_dist = np.inf
    for line in lines:
        sample = np.array(line.coords)
        d = cdist(p, sample).min()
        min_dist = min(min_dist, d)
    return float(min_dist)

def detect_vessels_in_sar_masked(
    sar_img,
    bbox,
    threshold_db=-12.0,
    lane_half_width_px=18,
    min_pixels=3,
    max_pixels=24,
    min_eccentricity=0.75,
    max_minor_axis=4.5,
    max_lane_distance_px=20,
    border_px=20,
    verbose=True,
):
    """
    Corridor-constrained SAR detector for Strait of Hormuz.

    Returns:
      clean_mask        : bool array of accepted detections
      vessel_count      : int accepted count
      detections_df     : dataframe of accepted object properties
      debug             : dict of intermediate masks/counts
    """
    if sar_img.ndim == 3:
        sar_img = sar_img[..., 0]

    h, w = sar_img.shape

    lane_mask, inbound_line, outbound_line = make_lane_mask(
        shape=(h, w),
        bbox=bbox,
        lane_half_width_px=lane_half_width_px,
    )
    border_mask = make_border_exclusion_mask((h, w), border_px=border_px)

    # 1) Raw threshold
    raw_mask = sar_img > threshold_db

    # 2) Restrict to lane corridor + avoid image edge clutter
    masked = raw_mask & lane_mask & border_mask

    # 3) Morphological cleanup
    masked = ndi.binary_opening(masked, structure=np.ones((2, 2)))
    masked = ndi.binary_closing(masked, structure=np.ones((2, 2)))
    masked = ndi.binary_fill_holes(masked)

    # 4) Connected components
    lbl = label(masked)
    props = regionprops(lbl, intensity_image=sar_img)

    accepted_labels = []
    rows = []

    for r in props:
        area = r.area
        ecc = float(getattr(r, "eccentricity", 0.0))
        major = float(getattr(r, "major_axis_length", 0.0))
        minor = float(getattr(r, "minor_axis_length", 0.0))
        mean_int = float(r.mean_intensity)
        dist_lane = region_distance_to_lines(r, [inbound_line, outbound_line])

        keep = (
            (min_pixels <= area <= max_pixels) and
            (ecc >= min_eccentricity) and
            (minor <= max_minor_axis) and
            (dist_lane <= max_lane_distance_px)
        )

        rows.append({
            "label": r.label,
            "area_px": area,
            "eccentricity": round(ecc, 3),
            "major_axis_px": round(major, 2),
            "minor_axis_px": round(minor, 2),
            "mean_db": round(mean_int, 2),
            "distance_to_lane_px": round(dist_lane, 2),
            "accepted": keep,
            "centroid_y": round(r.centroid[0], 2),
            "centroid_x": round(r.centroid[1], 2),
        })

        if keep:
            accepted_labels.append(r.label)

    clean_mask = np.isin(lbl, accepted_labels)
    detections_df = pd.DataFrame(rows).sort_values(
        ["accepted", "distance_to_lane_px", "area_px"],
        ascending=[False, True, True]
    ).reset_index(drop=True)

    debug = {
        "raw_threshold_count": int(label(raw_mask).max()),
        "lane_mask_pixels": int(lane_mask.sum()),
        "post_mask_component_count": int(label(masked).max()),
        "accepted_count": int(len(accepted_labels)),
    }

    if verbose:
        print("SAR masked detector diagnostics")
        print(f"  Raw threshold components     : {debug['raw_threshold_count']}")
        print(f"  Post-lane-mask components    : {debug['post_mask_component_count']}")
        print(f"  Accepted vessel-like targets : {debug['accepted_count']}")

    return clean_mask, len(accepted_labels), detections_df, debug


# -------------------------------------------------------------------
# Execution block
# -------------------------------------------------------------------
print("Fetching Sentinel-1 SAR imagery for masked vessel detection...")
sar_images = fetch_s1_sar_for_hormuz(HORMUZ_BBOX, days_back=4, label="Hormuz Strait")

sar_vessel_counts = []
sar_detection_tables = []
sar_debug_info = []

for i, img in enumerate(sar_images, start=1):
    img2d = img[..., 0] if img.ndim == 3 else img

    mask, n_vessels, det_df, dbg = detect_vessels_in_sar_masked(
        img2d,
        bbox=HORMUZ_BBOX,
        threshold_db=-12.0,
        lane_half_width_px=18,
        min_pixels=3,
        max_pixels=24,
        min_eccentricity=0.75,
        max_minor_axis=4.5,
        max_lane_distance_px=20,
        border_px=20,
        verbose=True,
    )

    sar_vessel_counts.append(n_vessels)
    sar_detection_tables.append(det_df)
    sar_debug_info.append(dbg)

    print(f"Scene {i}: accepted SAR vessel-like detections = {n_vessels}")
    if not det_df.empty:
        print(det_df.head(10).to_string(index=False))

if sar_vessel_counts:
    print(f"\nMedian masked SAR count: {int(np.median(sar_vessel_counts))}")
else:
    print("No SAR scenes fetched.")

🔧 CDSE Config Verification (explicit parameters):
  sh_base_url : https://sh.dataspace.copernicus.eu
  sh_token_url: https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token
  instance_id : ''
  client ID   : sh-cf7b5...
↔ Testing CDSE authentication...
✅ Auth OK: Found 10 collections
Sentinel-1 GRD available: True
Fetching Sentinel-1 SAR imagery for masked vessel detection...
  S1 SAR scenes fetched for Hormuz Strait: 1
SAR masked detector diagnostics
  Raw threshold components     : 826
  Post-lane-mask components    : 17
  Accepted vessel-like targets : 0
Scene 1: accepted SAR vessel-like detections = 0
 label  area_px  eccentricity  major_axis_px  minor_axis_px  mean_db  distance_to_lane_px  accepted  centroid_y  centroid_x
     1     13.0         0.418           4.30           3.90    -8.00                 9.52     False      237.77      520.38
    15    253.0         0.567          21.22          17.48    -5.35                20.60     False      42

#Sentinel-2 Optical Vessel Detection

> Add blockquote




In [ ]:
# Sentinel-2 Optical Vessel Detection
from typing import Any, Dict, List, Sequence, Tuple


def fetch_s2_vessels_hormuz(
    bbox_coords: Sequence[float],
    days_back: int = 5,
    max_cloud_pct: int = 60,
) -> List[Any]:
    """Fetch Sentinel-2 imagery and auxiliary masks for vessel detection.

    Args:
        bbox_coords: Bounding box in lon/lat order.
        days_back: Number of trailing days to search.
        max_cloud_pct: Maximum cloud percentage filter.

    Returns:
        List[Any]: Sentinel-2 response payloads for RGB, NDWI, and cloud masks.
    """
    end_dt: datetime = datetime.utcnow()
    start_dt: datetime = end_dt - timedelta(days=days_back)
    bbox_sh = BBox(bbox=bbox_coords, crs=CRS.WGS84)
    size: Tuple[int, int] = bbox_to_dimensions(bbox_sh, resolution=RESOLUTION)
    max_px: int = 1024
    if max(size) > max_px:
        scale: float = max(size) / max_px
        size = (int(size[0] / scale), int(size[1] / scale))

    evalscript: str = """
    //VERSION=3
    function setup() {
      return {
        input: ["B02", "B03", "B04", "B08", "SCL"],
        output: [
          { id: "rgb", bands: 3, sampleType: "UINT8" },
          { id: "ndwi", bands: 1, sampleType: "FLOAT32" },
          { id: "cloudfree", bands: 1, sampleType: "UINT8" }
        ]
      };
    }
    function evaluatePixel(samples) {
      var scl = samples.SCL;
      var isCloud = (scl === 8 || scl === 9 || scl === 10);
      var r = samples.B04;
      var g = samples.B03;
      var b = samples.B02;
      var nir = samples.B08;
      var ndwi = (g - nir) / (g + nir + 0.0001);
      var gain = 3.5;
      return {
        rgb: [
          Math.min(Math.round(r * gain * 255), 255),
          Math.min(Math.round(g * gain * 255), 255),
          Math.min(Math.round(b * gain * 255), 255)
        ],
        ndwi: [ndwi],
        cloudfree: [isCloud ? 0 : 255]
      };
    }
    """

    s2_cdse = DataCollection.SENTINEL2_L2A.define_from(
        "s2l2a_cdse",
        service_url=config.sh_base_url,
    )
    req = SentinelHubRequest(
        evalscript=evalscript,
        input_data=[SentinelHubRequest.input_data(
            data_collection=s2_cdse,
            time_interval=(start_dt.strftime("%Y-%m-%d"), end_dt.strftime("%Y-%m-%d")),
            maxcc=max_cloud_pct / 100,
        )],
        responses=[
            SentinelHubRequest.output_response("rgb", MimeType.PNG),
            SentinelHubRequest.output_response("ndwi", MimeType.TIFF),
            SentinelHubRequest.output_response("cloudfree", MimeType.TIFF),
        ],
        bbox=bbox_sh,
        size=size,
        config=config,
    )
    data: List[Any] = req.get_data()
    print(f"  ✅ S2 optical scenes fetched: {len(data)}")
    return data


def detect_vessels_optical(
    rgb_img: np.ndarray,
    ndwi_img: np.ndarray,
    min_brightness: int = 160,
) -> Tuple[np.ndarray, int]:
    """Detect bright vessel candidates over water in optical imagery.

    Args:
        rgb_img: RGB image array.
        ndwi_img: NDWI array used to isolate water pixels.
        min_brightness: Minimum average RGB brightness threshold.

    Returns:
        Tuple[np.ndarray, int]: Vessel mask and count of valid detections.
    """
    ndwi_2d: np.ndarray = ndwi_img[:, :, 0] if ndwi_img.ndim == 3 else ndwi_img
    water_mask: np.ndarray = ndwi_2d > 0.1
    brightness: np.ndarray = rgb_img.mean(axis=2) if rgb_img.ndim == 3 else rgb_img
    vessel_mask: np.ndarray = water_mask & (brightness > min_brightness)
    vessel_mask = ndi.binary_opening(vessel_mask, structure=np.ones((2, 2)))
    labeled, n_objects = ndi.label(vessel_mask)
    sizes = ndi.sum(vessel_mask, labeled, range(1, n_objects + 1))
    valid_ids: List[int] = [i + 1 for i, size in enumerate(sizes) if 2 <= size <= 50]
    clean_mask: np.ndarray = np.isin(labeled, valid_ids)
    return clean_mask, len(valid_ids)

#Layer Fusion

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# LAYER 3: Multi-Source Fusion (updated for 3 satellite sources)
# ─────────────────────────────────────────────────────────────────────────────
def fuse_vessel_estimates(ais_strait, ais_staging, sar_raw,
                          optical_raw=None, cloud_frac=None):
    """
    Fusion methodology:
    - SAR:     primary ground truth  → clean = raw × (1 - 0.15 FP rate)
    - Optical: supplementary         → cloud-weighted, detects smaller vessels
    - AIS:     lower bound only      → adjusted by dark-vessel factor
    Final fused = median of available clean estimates.
    """
    sar_clean    = int(sar_raw * (1 - SAR_FP_RATE))
    ais_total    = ais_strait + ais_staging
    ais_adj      = int(ais_total * DARK_VESSEL_FACTOR) if ais_total > 0 else None

    estimates    = {"SAR (clean)": sar_clean}
    if ais_adj:
        estimates["AIS (dark-adj)"] = ais_adj
    if optical_raw is not None and optical_raw > 0:
        # Weight optical by cloud fraction (less weight if cloudy)
        cloud_weight      = 1.0 - (cloud_frac or 0.0)
        optical_clean     = int(optical_raw * cloud_weight * (1 - 0.10))  # 10% optical FP
        estimates["Optical (clean)"] = optical_clean

    values  = [v for v in estimates.values() if v > 0]
    fused   = int(np.median(values)) if values else sar_clean
    pct_bl  = round((fused / BASELINE_DAILY) * 100, 1)

    print("\n" + "═"*58)
    print("  📡 MULTI-LAYER VESSEL ESTIMATE")
    print("═"*58)
    print(f"  SAR raw / clean          : {sar_raw} / {sar_clean}")
    if ais_total:
        print(f"  AIS visible (lower bound): {ais_total}  (×{DARK_VESSEL_FACTOR} → {ais_adj})")
    else:
        print(f"  AIS visible              : 0  (transponders off / GPS jamming)")
    if "Optical (clean)" in estimates:
        print(f"  Optical raw / clean      : {optical_raw} / {estimates['Optical (clean)']}  "
              f"(cloud={cloud_frac:.0%})")
    print("─"*58)
    print(f"  ➤ FUSED ESTIMATE         : ~{fused} vessels")
    print(f"  ➤ % of baseline          : {pct_bl}% of {BASELINE_DAILY}/day")
    print("═"*58)

    return fused, estimates, pct_bl



#Transit Time-Series from Historical AIS


# Comprehensive Dashboard

Uses dynamic history and runtime-derived live values.

In [ ]:
import plotly.graph_objects as go

# Comprehensive Dashboard
# Uses registry-driven history and runtime-derived overlay values

baseline_daily = BASELINE_DAILY
history = transit_history.copy().sort_values("date")
trend_y = history["vessels_portwatch"].combine_first(history["vessels_ais"])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=history["date"],
    y=trend_y,
    mode="lines+markers",
    name=DASHBOARD_CONFIG["history_series_name"],
    line=dict(color=ACCENT, width=2.5),
    marker=dict(size=5),
    fill="tozeroy",
    fillcolor=DASHBOARD_CONFIG["history_fill"],
    customdata=history["notes"],
    hovertemplate="%{x|%Y-%m-%d}<br>%{y:.0f} vessels<br>%{customdata}<extra></extra>",
))

if DASHBOARD_CONFIG["show_live_overlay"] and history["vessels_ais"].notna().any():
    live = history[history["vessels_ais"].notna()]
    fig.add_trace(go.Scatter(
        x=live["date"],
        y=live["vessels_ais"],
        mode="markers",
        name=DASHBOARD_CONFIG["live_series_name"],
        marker=dict(size=10, color=GREEN, symbol="diamond"),
        hovertemplate="%{x|%Y-%m-%d}<br>%{y:.0f} vessels (AIS live)<extra></extra>",
    ))

fig.add_hline(
    y=baseline_daily,
    line_dash="dash",
    line_color=MUTED,
    opacity=0.7,
    annotation_text=f"Baseline ~{baseline_daily}/day",
    annotation_position="top left",
)

fig.update_layout(
    template="plotly_dark",
    title=f"{DASHBOARD_CONFIG['title']} — updated {FETCH_TIME}",
    xaxis_title="Date",
    yaxis_title="Vessels",
    height=DASHBOARD_CONFIG["height"],
    paper_bgcolor=DARK_BG,
    plot_bgcolor=PANEL_BG,
    font=dict(color=TEXT),
    margin=dict(l=30, r=20, t=60, b=30),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)

fig.write_html("/content/hormuz_history_dashboard.html", include_plotlyjs="cdn")
fig

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import json
import os
from datetime import datetime

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML


# ── Guards ─────────────────────────────────────────────────────────────────────
if "transit_history" not in globals() or transit_history.empty:
    raise RuntimeError("transit_history is missing or empty. Run the dynamic history cell first.")


# ── Carrier fallback ───────────────────────────────────────────────────────────
if "df_carriers" not in globals():
    carrier_status = {
        "Maersk": {"status": "Suspended", "trapped": 14, "teu": 70000},
        "MSC": {"status": "Suspended", "trapped": 15, "teu": 109000},
        "CMA CGM": {"status": "Suspended", "trapped": 1, "teu": None},
        "Hapag-Lloyd": {"status": "Suspended", "trapped": 6, "teu": 25000},
        "COSCO": {"status": "Suspended", "trapped": 5, "teu": None},
        "ONE": {"status": "Suspended", "trapped": 147, "teu": 470000},
        "HMM": {"status": "Suspended", "trapped": 0, "teu": None},
        "Evergreen": {"status": "Suspended", "trapped": 0, "teu": None},
        "PIL": {"status": "Suspended", "trapped": 4, "teu": None},
    }
    df_carriers = pd.DataFrame(carrier_status).T.reset_index()
    df_carriers.columns = ["Carrier", "Status", "Vessels Trapped", "TEU Trapped"]


# ── Canonical history ─────────────────────────────────────────────────────────
history = transit_history.copy().sort_values("date").reset_index(drop=True)
history["date"] = pd.to_datetime(history["date"]).dt.tz_localize(None)

if "vessels_portwatch" not in history.columns:
    history["vessels_portwatch"] = history["ntotal"] if "ntotal" in history.columns else np.nan

if "vessels_ais" not in history.columns:
    history["vessels_ais"] = np.nan

if "notes" not in history.columns:
    history["notes"] = ""

if "sar_vessel_counts" in globals() and sar_vessel_counts:
    sar_clean = int(round(np.median(sar_vessel_counts) * (1 - SAR_FP_RATE)))
else:
    sar_clean = 0

sar_scene_x = None
sar_scene_y = None
if "sar_vessel_counts" in globals() and sar_vessel_counts:
    sar_scene_y = [int(round(v * (1 - SAR_FP_RATE))) for v in sar_vessel_counts]
    n = len(sar_scene_y)
    if n <= len(history):
        sar_scene_x = history["date"].tail(n).tolist()
    else:
        sar_scene_x = pd.date_range(end=history["date"].iloc[-1], periods=n, freq="D").tolist()

latest_row = history.iloc[-1]
latest_transit_rate = latest_row.get("vessels_portwatch", np.nan)
latest_transit_source = "PortWatch"
if pd.isna(latest_transit_rate) and pd.notna(latest_row.get("vessels_ais", np.nan)):
    latest_transit_rate = latest_row["vessels_ais"]
    latest_transit_source = "AIS live overlay"

latest_transit_rate = int(round(float(latest_transit_rate))) if pd.notna(latest_transit_rate) else 0
latest_transit_pct = round(latest_transit_rate / BASELINE_DAILY * 100, 1) if BASELINE_DAILY else np.nan

live_ais_count = int(history["vessels_ais"].dropna().iloc[-1]) if history["vessels_ais"].notna().any() else 0

ais_adj = 0
if "hormuz_vessels" in globals() and "staging_vessels" in globals():
    ais_adj = int(round((len(hormuz_vessels) + len(staging_vessels)) * DARK_VESSEL_FACTOR))

occupancy_breakdown = {}
if sar_clean > 0:
    occupancy_breakdown["SAR occupancy (clean)"] = sar_clean
if ais_adj > 0:
    occupancy_breakdown["AIS occupancy (dark-adj)"] = ais_adj

FUSED = latest_transit_rate
PCT_BL = latest_transit_pct


# ── Commodity helpers ─────────────────────────────────────────────────────────
def safe_global(name, default):
    return globals().get(name, default)

commodity_labels_local = safe_global(
    "commodity_labels",
    ["Brent Crude", "WTI Crude", "EU Gas (TTF)", "US Gasoline"],
)
commodity_pct_local = safe_global(
    "commodity_pct",
    [
        safe_global("brent_pct_change", 0),
        safe_global("wti_pct_change", 0),
        safe_global("ttf_pct_change", 0),
        safe_global("gasoline_pct_change", 0),
    ],
)


# ── Styling helper ────────────────────────────────────────────────────────────
def dark_layout(legend_override=None, **kwargs):
    base_legend = dict(
        bgcolor=PANEL_BG,
        bordercolor="#30363d",
        borderwidth=1,
        font=dict(color=TEXT, size=11),
    )
    if legend_override:
        base_legend.update(legend_override)

    layout = dict(
        paper_bgcolor=DARK_BG,
        plot_bgcolor=PANEL_BG,
        font=dict(color=TEXT, family="Inter,system-ui,sans-serif", size=12),
        margin=dict(l=50, r=24, t=60, b=40),
        legend=base_legend,
    )
    layout.update(kwargs)
    return layout


# ── FIG 1: Transit timeline ───────────────────────────────────────────────────
fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=history["date"],
    y=history["vessels_portwatch"].combine_first(history["vessels_ais"]),
    mode="lines+markers",
    name="Observed transit rate",
    line=dict(color=ACCENT, width=2.5),
    marker=dict(size=5),
    fill="tozeroy",
    fillcolor=DASHBOARD_CONFIG.get("history_fill", "rgba(46,169,223,0.10)"),
    customdata=history["notes"],
    hovertemplate="%{x|%Y-%m-%d}<br>%{y:.0f} transits/day<br>%{customdata}<extra></extra>",
))

if sar_scene_x is not None and sar_scene_y is not None:
    fig1.add_trace(go.Scatter(
        x=sar_scene_x,
        y=sar_scene_y,
        mode="markers+lines",
        name="SAR occupancy (clean)",
        line=dict(color=WARN, width=2, dash="dot"),
        marker=dict(size=6, color=WARN, symbol="circle-open"),
        hovertemplate="%{x|%Y-%m-%d}<br>%{y:.0f} SAR-clean occupancy<extra></extra>",
    ))

live_rows = history[history["vessels_ais"].notna()]
if DASHBOARD_CONFIG.get("show_live_overlay", True) and not live_rows.empty:
    fig1.add_trace(go.Scatter(
        x=live_rows["date"],
        y=live_rows["vessels_ais"],
        mode="markers",
        name="AIS live overlay",
        marker=dict(size=10, color=GREEN, symbol="diamond"),
        hovertemplate="%{x|%Y-%m-%d}<br>%{y:.0f} vessels (live AIS snapshot)<extra></extra>",
    ))

fig1.add_hline(
    y=BASELINE_DAILY,
    line_dash="dash",
    line_color=MUTED,
    opacity=0.8,
    annotation_text=f"Baseline ~{int(BASELINE_DAILY)}/day",
    annotation_position="top left",
)

fig1.update_layout(**dark_layout(
    height=390,
    title=dict(
        text=f"{DASHBOARD_CONFIG.get('title', 'Strait of Hormuz Transit History')} — updated {FETCH_TIME}",
        font=dict(color=TEXT, size=14),
    ),
    xaxis=dict(gridcolor="#21262d", title="Date", tickfont=dict(color=TEXT)),
    yaxis=dict(gridcolor="#21262d", title="Daily transits", tickfont=dict(color=TEXT)),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
))
fig1.update_traces(cliponaxis=False)


# ── FIG 2: Transit KPI + occupancy ────────────────────────────────────────────
fig2 = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.42, 0.58],
    subplot_titles=["Transit rate (% of baseline)", "Occupancy context"],
    specs=[[{"type": "indicator"}, {"type": "bar"}]],
)

fig2.add_trace(go.Indicator(
    mode="gauge+number+delta",
    value=PCT_BL,
    delta={"reference": 100, "valueformat": ".1f", "suffix": "%"},
    title={"text": f"~{FUSED} transits/day", "font": {"color": TEXT, "size": 14}},
    gauge={
        "axis": {"range": [0, 100], "tickcolor": MUTED, "tickfont": {"color": TEXT}},
        "bar": {"color": DANGER},
        "steps": [
            {"range": [0, 10], "color": "#2d1b1b"},
            {"range": [10, 40], "color": "#2d2210"},
            {"range": [40, 70], "color": "#1b2d1e"},
            {"range": [70, 100], "color": "#1b221b"},
        ],
        "threshold": {"line": {"color": GREEN, "width": 2}, "thickness": 0.8, "value": 100},
        "bgcolor": PANEL_BG,
    },
    number={"suffix": "% baseline", "font": {"color": TEXT, "size": 16}},
), row=1, col=1)

srcs = list(occupancy_breakdown.keys()) if occupancy_breakdown else ["Latest observed transit"]
vals = list(occupancy_breakdown.values()) if occupancy_breakdown else [latest_transit_rate]
colors_bar = [WARN if "SAR" in s else MUTED for s in srcs]

fig2.add_trace(go.Bar(
    x=srcs,
    y=vals,
    marker_color=colors_bar,
    text=[str(v) for v in vals],
    textposition="outside",
    textfont=dict(color=TEXT),
    showlegend=False,
    hovertemplate="%{x}: <b>%{y}</b><extra></extra>",
), row=1, col=2)

fig2.update_layout(**dark_layout(
    height=310,
    margin=dict(l=10, r=10, t=50, b=30),
    yaxis2=dict(gridcolor="#21262d", title_text="Count", range=[0, max(vals) + 5], tickfont=dict(color=TEXT)),
))
for ann in fig2.layout.annotations:
    ann.font.color = TEXT


# ── FIG 3: Vessel composition ──────────────────────────────────────────────────
type_counts = {}
if "df_ais" in globals() and not df_ais.empty and "type_name" in df_ais.columns:
    type_counts = df_ais["type_name"].fillna("Unknown").value_counts().head(6).to_dict()
if not type_counts:
    type_counts = {
        "Tanker": 6,
        "Cargo": 5,
        "Bulk Carrier": 4,
        "General Cargo": 3,
        "Military": 2,
        "Other": 2,
    }

pie_colors = [DANGER, WARN, "#ffcc00", ACCENT, GREEN, MUTED][:len(type_counts)]

fig3 = go.Figure(go.Pie(
    labels=list(type_counts.keys()),
    values=list(type_counts.values()),
    marker=dict(colors=pie_colors, line=dict(color=DARK_BG, width=1.5)),
    textfont=dict(color=TEXT, size=10),
    hole=0.35,
    hovertemplate="%{label}: <b>%{value}</b> vessels (%{percent})<extra></extra>",
))
fig3.update_layout(**dark_layout(
    height=290,
    title=dict(text="Current AIS Vessel Types", font_color=TEXT, font_size=12),
    showlegend=True,
    legend=dict(orientation="v", x=1.0, y=0.5, bgcolor=PANEL_BG, bordercolor="#30363d", borderwidth=1),
    margin=dict(l=10, r=140, t=45, b=10),
))
fig3.update_layout(uniformtext_minsize=9, uniformtext_mode="hide")


# ── FIG 4: Energy impact ──────────────────────────────────────────────────────
fig4 = go.Figure(go.Bar(
    x=commodity_labels_local,
    y=commodity_pct_local,
    marker_color=[DANGER if p > 50 else WARN for p in commodity_pct_local],
    text=[f"{p:+.0f}%" for p in commodity_pct_local],
    textposition="outside",
    textfont=dict(color=TEXT, size=12),
    hovertemplate="%{x}: <b>%{y:+.1f}%</b><extra></extra>",
))
fig4.update_layout(**dark_layout(
    height=290,
    title=dict(text="Energy Price Impact vs Pre-Crisis", font_color=TEXT, font_size=12),
    margin=dict(l=40, r=20, t=45, b=40),
    showlegend=False,
    xaxis=dict(gridcolor="#21262d", tickfont=dict(color=TEXT)),
    yaxis=dict(gridcolor="#21262d", title_text="% change", tickfont=dict(color=TEXT)),
))
fig4.update_traces(cliponaxis=False)


# ── FIG 5: Carrier table ──────────────────────────────────────────────────────
teu_labels = [f"{int(t):,}" if not pd.isna(t) else "N/A" for t in df_carriers["TEU Trapped"]]

fig5 = go.Figure(go.Table(
    header=dict(
        values=["<b>Carrier</b>", "<b>Status</b>", "<b>Vessels</b>", "<b>TEU Trapped</b>"],
        fill_color="#21262d",
        font=dict(color=TEXT, size=12),
        align="left",
        line_color="#30363d",
        height=34,
    ),
    cells=dict(
        values=[
            df_carriers["Carrier"],
            df_carriers["Status"],
            df_carriers["Vessels Trapped"],
            teu_labels,
        ],
        fill_color=[[PANEL_BG if i % 2 == 0 else "#1c2128" for i in range(len(df_carriers))]],
        font_color=[[TEXT] * len(df_carriers)],
        align="left",
        height=29,
        font=dict(size=11),
    ),
))
fig5.update_layout(**dark_layout(
    height=310,
    margin=dict(l=0, r=0, t=10, b=0),
    title=dict(text="Carrier Suspension Status", font_color=TEXT, font_size=12),
))


# ── FIG 6: Simple schematic map ───────────────────────────────────────────────
fig6 = go.Figure()
np.random.seed(42)
staging_lats = [25.65 + np.random.uniform(-0.2, 0.2) for _ in range(18)]
staging_lons = [56.70 + np.random.uniform(-0.4, 0.3) for _ in range(18)]
inbound_lats = [26.50, 26.48, 26.45, 26.42, 26.38, 26.33]
inbound_lons = [56.50, 56.60, 56.80, 57.00, 57.20, 57.40]
outbound_lats = [26.18, 26.22, 26.28, 26.32, 26.28, 26.22]
outbound_lons = [55.90, 56.10, 56.30, 56.50, 56.70, 56.90]
corridor_lats = [26.40, 26.58, 26.65, 26.62, 26.55, 26.45]
corridor_lons = [55.90, 56.10, 56.40, 56.70, 57.00, 57.20]

fig6.add_trace(go.Scattergeo(
    lat=staging_lats, lon=staging_lons,
    mode="markers",
    marker=dict(size=8, color=MUTED, opacity=0.8, symbol="square"),
    name="GoO staging",
    hovertemplate="Staging vessel<br>%{lat:.2f}N %{lon:.2f}E<extra></extra>",
))
fig6.add_trace(go.Scattergeo(
    lat=corridor_lats, lon=corridor_lons,
    mode="lines+markers",
    line=dict(color=WARN, width=3),
    marker=dict(size=5, color=WARN),
    name="Tehran-approved corridor",
    hovertemplate="Tehran corridor<extra></extra>",
))
fig6.add_trace(go.Scattergeo(
    lat=inbound_lats, lon=inbound_lons,
    mode="lines",
    line=dict(color=ACCENT, width=2, dash="dot"),
    name="Normal inbound lane",
    hovertemplate="Normal inbound lane<extra></extra>",
))
fig6.add_trace(go.Scattergeo(
    lat=outbound_lats, lon=outbound_lons,
    mode="lines",
    line=dict(color=GREEN, width=2, dash="dot"),
    name="Normal outbound lane",
    hovertemplate="Normal outbound lane<extra></extra>",
))
fig6.add_trace(go.Scattergeo(
    lat=[26.50], lon=[56.10],
    mode="markers+text",
    marker=dict(size=14, color=DANGER, symbol="star"),
    text=["IRGC"],
    textposition="top right",
    textfont=dict(color=DANGER, size=11),
    name="IRGC patrol zone",
    hovertemplate="IRGC patrol area<extra></extra>",
))
fig6.update_layout(**dark_layout(
    legend_override=dict(
        x=0.01, y=0.01, xanchor="left", yanchor="bottom",
        bgcolor="rgba(22,27,34,0.85)", bordercolor="#30363d", borderwidth=1,
        font=dict(color=TEXT, size=10),
    ),
    height=420,
    margin=dict(l=0, r=0, t=45, b=0),
    showlegend=True,
    title=dict(text="Strait of Hormuz Schematic", font_color=TEXT, font_size=12),
    geo=dict(
        lonaxis=dict(range=[54.5, 58.5]),
        lataxis=dict(range=[24.5, 27.8]),
        showland=True,
        landcolor="#2d3b2a",
        showocean=True,
        oceancolor="#0a1628",
        showlakes=True,
        lakecolor="#0a1628",
        showcountries=True,
        countrycolor="#4a5568",
        showcoastlines=True,
        coastlinecolor="#5a6a7a",
        showframe=False,
        bgcolor=DARK_BG,
        resolution=50,
        projection=dict(type="mercator"),
    ),
))


# ── KPI HTML ───────────────────────────────────────────────────────────────────
if "brent_label" in globals():
    brent_label_local = brent_label
else:
    brent_label_local = "N/A"

if "brent_sub" in globals():
    brent_sub_local = brent_sub
else:
    brent_sub_local = "Brent unavailable"

if "brent_color" in globals():
    brent_color_local = brent_color
else:
    brent_color_local = WARN

if "war_risk_label" in globals():
    war_risk_label_local = war_risk_label
else:
    war_risk_label_local = "N/A"

if "war_risk_sub" in globals():
    war_risk_sub_local = war_risk_sub
else:
    war_risk_sub_local = "War-risk unavailable"


def kpi(title, val, sub, color):
    return f"""
    <div class="kpi">
      <div class="kpi-label">{title}</div>
      <div class="kpi-value" style="color:{color}">{val}</div>
      <div class="kpi-sub">{sub}</div>
    </div>
    """

teu_total = int(df_carriers["TEU Trapped"].dropna().sum())

kpis_html = "".join([
    kpi("Transit Rate", str(latest_transit_rate), f"{latest_transit_pct:.1f}% of baseline · {latest_transit_source}", DANGER),
    kpi("Baseline Comparison", f"{PCT_BL:.1f}%", f"{FUSED} transits/day vs {int(BASELINE_DAILY)}/day", DANGER),
    kpi("SAR Occupancy",str(sar_clean) if ("sar_vessel_counts" in globals() and sar_vessel_counts) else "N/A",
    "Instantaneous scene count, not daily flow" if ("sar_vessel_counts" in globals() and sar_vessel_counts) else "No SAR input available",
    WARN,),
    kpi("AIS Visible Today", str(live_ais_count), "Lower bound only", MUTED),
    kpi("Carriers Suspended", f"{len(df_carriers)}/9", "Major lines", DANGER),
    kpi("Vessels Trapped", str(int(df_carriers["Vessels Trapped"].sum())), "Unable to transit", DANGER),
    kpi("TEU Trapped", f"{round(teu_total / 1000):,}K", "Estimated capacity", WARN),
    kpi("Days Disrupted", str(DAYS_DISRUPTED), f"Since {CRISIS_START.strftime('%b %d %Y')}", WARN),
    kpi("Brent Crude", brent_label_local, brent_sub_local, brent_color_local),
    kpi("War Risk Insurance", war_risk_label_local, war_risk_sub_local, WARN),
])


# ── Assemble HTML ─────────────────────────────────────────────────────────────
snapshot_ts = datetime.utcnow().strftime("%Y-%m-%d %H:%MZ")

html_template = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Hormuz Vessel Reconnaissance Dashboard</title>
<script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap" rel="stylesheet">
<style>
*,*::before,*::after{{box-sizing:border-box;margin:0;padding:0}}
body{{background:{dark_bg};color:{text};font-family:Inter,system-ui,sans-serif;font-size:13px;line-height:1.5}}
header{{background:{panel_bg};border-bottom:1px solid #30363d;padding:14px 24px;display:flex;align-items:center;gap:14px;position:sticky;top:0;z-index:10}}
header h1{{font-size:17px;font-weight:700}}
.badge{{background:{danger}22;color:{danger};border:1px solid {danger}66;border-radius:4px;padding:2px 9px;font-size:11px;font-weight:600;letter-spacing:.04em}}
.ts{{color:{muted};font-size:11px;margin-left:auto}}
.kpi-row{{display:flex;gap:10px;padding:14px 20px;flex-wrap:wrap}}
.kpi{{background:{panel_bg};border:1px solid #30363d;border-radius:8px;padding:14px 18px;flex:1;min-width:160px}}
.kpi-label{{color:{muted};font-size:10px;text-transform:uppercase;letter-spacing:.06em;margin-bottom:4px}}
.kpi-value{{font-size:24px;font-weight:700;line-height:1.1}}
.kpi-sub{{color:{muted};font-size:10px;margin-top:4px}}
.grid2{{display:grid;grid-template-columns:1fr 1fr;gap:12px;padding:0 20px 12px}}
.full{{padding:0 20px 12px}}
.panel{{background:{panel_bg};border:1px solid #30363d;border-radius:8px;overflow:hidden}}
.section-tag,.layer-badge{{display:inline-block;color:{muted};font-size:10px;text-transform:uppercase;letter-spacing:.08em;padding:8px 14px 0;font-weight:500}}
.layer-badge{{background:#21262d;border:1px solid #30363d;border-radius:4px;padding:1px 7px;margin:8px 14px 0;letter-spacing:.04em}}
@media (max-width:768px){{.grid2{{grid-template-columns:1fr}}}}
</style>
</head>
<body>
<header>
  <svg width="22" height="22" viewBox="0 0 24 24" fill="none" stroke="{accent}" stroke-width="2" stroke-linecap="round" stroke-linejoin="round">
    <path d="M3 17l4-8 4 4 4-6 4 10"></path>
    <path d="M3 21h18"></path>
  </svg>
  <h1>Hormuz Vessel Reconnaissance</h1>
  <span class="badge">CRISIS ACTIVE</span>
  <span class="ts">Snapshot {snapshot_ts}</span>
</header>
<div class="kpi-row">{kpis_html}</div>
<div class="full"><span class="layer-badge">Historical transit</span><div class="panel" id="fig1"></div></div>
<div class="grid2"><div><span class="layer-badge">Transit KPI + occupancy context</span><div class="panel" id="fig2"></div></div><div><span class="section-tag">Carrier suspension status</span><div class="panel" id="fig5"></div></div></div>
<div class="grid2"><div><span class="layer-badge">Current AIS vessel composition</span><div class="panel" id="fig3"></div></div><div><span class="section-tag">Energy price impact</span><div class="panel" id="fig4"></div></div></div>
<div class="full" style="padding-bottom:24px"><span class="layer-badge">Strait schematic</span><div class="panel" id="fig6"></div></div>
<script>
const cfg = {{responsive:true, displayModeBar:false}};
Plotly.newPlot('fig1', {fig1_json}, cfg);
Plotly.newPlot('fig2', {fig2_json}, cfg);
Plotly.newPlot('fig3', {fig3_json}, cfg);
Plotly.newPlot('fig4', {fig4_json}, cfg);
Plotly.newPlot('fig5', {fig5_json}, cfg);
Plotly.newPlot('fig6', {fig6_json}, cfg);
</script>
</body>
</html>
"""

html = html_template.format(
    dark_bg=DARK_BG,
    panel_bg=PANEL_BG,
    text=TEXT,
    accent=ACCENT,
    danger=DANGER,
    muted=MUTED,
    snapshot_ts=snapshot_ts,
    kpis_html=kpis_html,
    fig1_json=fig1.to_json(),
    fig2_json=fig2.to_json(),
    fig3_json=fig3.to_json(),
    fig4_json=fig4.to_json(),
    fig5_json=fig5.to_json(),
    fig6_json=fig6.to_json(),
)

os.makedirs("output", exist_ok=True)
with open("output/hormuz_dashboard.html", "w", encoding="utf-8") as f:
    f.write(html)

print("✅ Saved: output/hormuz_dashboard.html")
display(HTML(html))

✅ Saved: output/hormuz_dashboard.html


# Save Data: Export transit CSV + AIS snapshot


In [ ]:
# Save transit history
transit_history.to_csv("output/hormuz_transit_history.csv", index=False)

# Save current AIS snapshot
if not df_ais.empty:
    df_ais.to_csv("output/hormuz_ais_snapshot.csv", index=False)
    print(f"AIS snapshot: {len(df_ais)} vessels → output/hormuz_ais_snapshot.csv")

print(f"Transit history (95 days) → output/hormuz_transit_history.csv")
print(f"\n{'='*55}")
print(f"  SUMMARY — {TODAY.strftime('%Y-%m-%d')}")
print(f"  Crisis onset    : 2026-02-28 (Day {DAYS_DISRUPTED})")
print(f"  Strait status   : Closed to most commercial shipping")
print(f"  AIS daily count : ~{transit_history.iloc[-1]['vessels_ais']} vessels (lower bound)")
print(f"  SAR est.        : ~8–12 vessels/day (incl. dark ships)")
print(f"  Approved corridor: Iran-guided route via Larak/Qeshm")
print(f"{'='*55}")




This factor is a scenario constant, not a measurement. It is intended to produce
a defensible upper-bound-style correction for partially darkened traffic without
assuming a full blockade or complete sensor collapse.


### Limitations


- AIS is a self-reported system — position spoofing is trivially achievable and
  has been documented in regional gray-zone environments
- S-AIS satellite revisit introduces latency and intermittent coverage gaps for
  vessels outside coastal VHF range
- The dark vessel factor is an estimate, not a measurement; actual ratio varies
  by vessel class, route, and threat perception
- AIS data from aisstream.io is aggregated from multiple receivers with variable
  coverage — signal gaps in narrow channels are possible


---


## Layer 3 — Sentinel-2 Optical (Cloud-Permitting Validation)


### What it measures


Sentinel-2 MSI (MultiSpectral Instrument) captures 13 spectral bands at 10–60 m
resolution in the visible and near-infrared. Unlike SAR, optical imagery requires
daylight and cloud-free conditions. At 10 m resolution, vessels ≥20 m are
detectable as bright targets against dark water, with wake patterns sometimes
helping infer speed and heading. Recent Sentinel-2 vessel-detection work relies
increasingly on learned detectors rather than simple thresholding, especially in
coastal and high-density waters.


### Implementation


- **Bands**: B04 (Red, 10 m) + B08 (NIR, 10 m) — high contrast for bright hulls
  against dark water; NIR helps suppress some sunglint effects
- **Cloud masking**: SCL (Scene Classification Layer) applied; pixels classified
  as cloud, cloud shadow, or saturated excluded before detection
- **Detection**: deep-learning or learned object-detection output preferred where
  available; otherwise NDWI / reflectance gating can be used as a fallback
- **Blob filter**: 1–20 pixels at 10 m = 10 m–200 m (wider than SAR lower bound
  to capture smaller craft visible in optical)
- **False positive rate**: 10% (optical_fp_rate = 0.10) — whitecaps, sunglint,
  and specular highlights are the main confounders
- **Revisit**: 5-day repeat at Hormuz latitude; cloud-free scenes may be infrequent
  in summer months
- **Wire-in point**: `optical_count=None` parameter in `fuse_vessel_estimates()`
  — pass `optical_clean` when a cloud-free scene is available


### Limitations


- Entirely unavailable in cloud cover, dust haze, or night-time
- Sunglint at certain solar angles can saturate B04 pixels and mask vessels
- Cannot detect submerged vessels or vessels hidden by atmospheric haze
- 5-day revisit means optical may be hours or days stale relative to SAR/AIS


---


## Fusion Layer — Multi-Source Estimate


### Methodology


The fusion function `fuse_vessel_estimates()` implements a median-of-estimates
approach rather than a weighted average. The median is chosen over mean because:


1. It is robust to a single outlier layer, such as SAR over-detection during
   high sea state without a corresponding AIS signal
2. It makes no assumption about the relative accuracy of each sensor
3. It degrades gracefully when optical is unavailable (median of two values)


```python
fused_count = median([SAR_clean, AIS_dark_adjusted, optical_clean])
```


When only two sources are available (optical absent), the median equals the
lower of the two values — a deliberately conservative choice appropriate for a
crisis scenario where over-reporting vessel activity is more dangerous than
under-reporting.


### Sensor capability matrix


| Capability | AIS | SAR | Optical |
|---|---|---|---|
| Detects dark vessels | ✗ | ✅ | ✅ |
| All-weather | ✅ | ✅ | ✗ |
| Night-time | ✅ | ✅ | ✗ |
| Position spoofing resistant | ✗ | ✅ | ✅ |
| Sub-20m vessels | ✅ | Partial | Partial |
| Vessel identity | ✅ | ✗ | ✗ |
| Real-time (<1 min latency) | ✅ | ✗ (6h–12h) | ✗ (days) |
| Free / no-auth | ✅ | ✅ (CDSE) | ✅ (CDSE) |


### Calibration constants


| Constant | Value | Derivation |
|---|---|---|
| `SAR_FP_RATE` | 0.15 | Conservative midpoint after masking and thresholding |
| `DARK_VESSEL_FACTOR` | 2.5× | Crisis scenario constant for AIS-dark uplift |
| `OPTICAL_FP_RATE` | 0.10 | Cloud-masked optical validation assumption |
| `BASELINE_DAILY` | 151 vessels/day | Historical PortWatch baseline used for normalization |


### Output interpretation


The fused estimate carries an implicit uncertainty of roughly ±20–30% due to the
cumulative uncertainty of the three sensor streams. It should be interpreted as
an order-of-magnitude constraint rather than a precise count. For operational
intelligence, confidence is increased when all three layers agree within 15% of
each other; divergence above 50% warrants manual review of the SAR scene for
clutter artefacts or the AIS data for spoofing and dropouts.


---


## Data Sources


- Sentinel-1 SAR imagery: ESA / Copernicus Data Space Ecosystem (CDSE),
  `sh.dataspace.copernicus.eu`
- Sentinel-2 optical imagery: ESA / CDSE, same endpoint
- AIS data: aisstream.io WebSocket v0 stream (S-AIS aggregator)
- Dark vessel factor methodology: crisis-era maritime intelligence heuristics
  informed by commercial monitoring and sanctions-evasion patterns
- SAR vessel detection: Sentinel-1 ship detection literature, CFAR / local-threshold
  workflows, and recent open-source narrow-corridor monitoring methods
- Optical vessel detection: Sentinel-2 vessel detection literature and deep-learning
  detectors validated on 10 m imagery
- False positive rates: SAR clutter / sidelobe literature and optical whitecap
  confound studies

# Down the rabbit hole



*   Polarimetry: https://science.nasa.gov/mission/nisar/polarimetry/
*   MarineTraffic tracker: https://www.marinetraffic.com/en/ais/home/centerx:49.9/centery:26.7/zoom:8

